# Hot-Swap: Transparently Replace pyccl with an Emulator

This notebook demonstrates the hot-swap machinery: train an emulator,
register it, and have computation calls automatically route through the
emulator instead of pyccl — with no changes to the calling code.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

from tissage_cosmique.computations.distances import comoving_angular_distance
from tissage_cosmique.emulators import GPEmulator, build_training_data, params_to_feature_matrix
from tissage_cosmique.swap import swappable, register, original, reset, list_registered

## 1. Train an emulator

In [ ]:
PARAM_NAMES = ["Omega_c", "h", "sigma8"]
FIXED = dict(Omega_b=0.0486, n_s=0.9667, Omega_k=0.0, w0=-1.0, wa=0.0)
a_grid = np.linspace(0.2, 0.8, 50)

rng = np.random.default_rng(42)
train_samples = [
    {"Omega_c": rng.uniform(0.22, 0.32), "h": rng.uniform(0.62, 0.75),
     "sigma8": rng.uniform(0.77, 0.87), **FIXED}
    for _ in range(50)
]

X, y = build_training_data(comoving_angular_distance, train_samples, a_grid, param_names=PARAM_NAMES)
emu = GPEmulator(feature_names=PARAM_NAMES + ["a"])
emu.fit(X, y)
print(f"Emulator trained: R2={emu.metadata['training_score']:.6f}")

## 2. Create a swappable version and register the emulator

In [ ]:
# Wrap the computation
compute = swappable(comoving_angular_distance)

# Before registration — calls go to pyccl
test_params = {"Omega_c": 0.27, "h": 0.68, "sigma8": 0.81, **FIXED}
result_pyccl = compute(test_params, a_grid)
print(f"Before registration: compute() uses pyccl -> result[0] = {result_pyccl[0]:.2f} Mpc")

# Register the emulator
register(comoving_angular_distance, emu, PARAM_NAMES)
print(f"\nRegistered: {list_registered()}")

# After registration — calls go to the emulator
result_emu = compute(test_params, a_grid)
print(f"After registration:  compute() uses emulator -> result[0] = {result_emu[0]:.2f} Mpc")
print(f"Difference: {abs(result_pyccl[0] - result_emu[0]):.4f} Mpc")

## 3. The `original()` context manager

Temporarily bypass the emulator to get the exact pyccl result — useful for
validation or when you need ground truth.

In [ ]:
# Inside the context: pyccl
with original(comoving_angular_distance):
    result_truth = compute(test_params, a_grid)

# Outside the context: emulator again
result_swapped = compute(test_params, a_grid)

print(f"with original(): {result_truth[0]:.2f} Mpc (pyccl)")
print(f"without:         {result_swapped[0]:.2f} Mpc (emulator)")

## 4. Speed comparison

In [ ]:
n_calls = 100

# Emulator (swapped)
t0 = time.time()
for _ in range(n_calls):
    compute(test_params, a_grid)
emu_time = time.time() - t0

# pyccl (original)
with original(comoving_angular_distance):
    t0 = time.time()
    for _ in range(n_calls):
        compute(test_params, a_grid)
    pyccl_time = time.time() - t0

print(f"Emulator: {n_calls} calls in {emu_time:.3f}s ({emu_time/n_calls*1000:.1f} ms/call)")
print(f"pyccl:    {n_calls} calls in {pyccl_time:.3f}s ({pyccl_time/n_calls*1000:.1f} ms/call)")
print(f"Speedup:  {pyccl_time/emu_time:.1f}x")

## 5. Visual comparison across scale factors

In [ ]:
with original(comoving_angular_distance):
    truth = compute(test_params, a_grid)

pred = compute(test_params, a_grid)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(a_grid, truth, "k-", lw=2, label="pyccl (original)")
ax.plot(a_grid, pred, "r--", lw=1.5, label="emulator (swapped)")
ax.set_xlabel("Scale factor a")
ax.set_ylabel("Distance [Mpc]")
ax.set_title("Hot-swapped computation")
ax.legend()

ax = axes[1]
rel_err = np.abs(pred - truth) / np.maximum(np.abs(truth), 1.0) * 100
ax.plot(a_grid, rel_err, "b-", lw=1.5)
ax.set_xlabel("Scale factor a")
ax.set_ylabel("Relative error [%]")
ax.set_title("Emulator error")

plt.tight_layout()
plt.show()

In [ ]:
# Clean up
reset()
print("Registry cleared")